# 📥 EWIT – Lấy bản ghi Zoom mới + AI đề xuất cắt
Lớp được xác định từ tab **Danh sách lớp** của Sheet điều khiển (Zoom ID + khoảng ngày + giờ bắt đầu). Thư mục: `MãKhóa-TàiKhoản-ZoomID` (ví dụ `EG2-01-CLB-936 3686 9490`); buổi chưa có trong danh sách → `CHUA-XEP-…`.

Khóa Zoom đọc từ file **`.env`** nằm cùng thư mục với Sheet điều khiển, dạng:
```
ZOOM_CLB_ACCOUNT_ID=...
ZOOM_CLB_CLIENT_ID=...
ZOOM_CLB_CLIENT_SECRET=...
```
(chỉ 1 tài khoản thì `ZOOM_ACCOUNT_ID=…` hoặc `ACCOUNT_ID=…` cũng được).

Khóa AI (một trong các khóa, tuỳ nhà cung cấp chọn ở tab **Cấu hình**): `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENROUTER_API_KEY`.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess, sys
subprocess.run('rm -rf /content/VideoApps && git clone -q --depth 1 -b claude/blissful-thompson-ey00yo '
               'https://github.com/tnho113/VideoApps /content/VideoApps', shell=True, check=True)
sys.path.insert(0, '/content/VideoApps/tools/zoom_cut')
for m in ['sheet_job', 'zoom_sync', 'ai_cut', 'transcript_compact']: sys.modules.pop(m, None)
from zoom_sync import ZoomSync
TAI_KHOAN = ['CLB']   # ✏️ mã tài khoản Zoom (khớp cột 'Tài khoản Zoom' ở tab Danh sách lớp)
zs = ZoomSync()
ENV = zs.load_env()
ACCOUNTS = zs.accounts_from_env(TAI_KHOAN, ENV)
print('✅ Sẵn sàng – tài khoản đã nạp:', list(ACCOUNTS))

## Bước 1 – Xem trước bản ghi Zoom mới (không tải, không ghi Sheet)

In [ ]:
zs.run(ACCOUNTS, dry_run=True)

## Bước 1b – Lấy thật ✏️
`GIOI_HAN = 1` để chạy thử 1 buổi; `None` = lấy hết. Mỗi buổi tạo 1 thư mục trong *Video Zoom (tự động)* và 1 dòng ở tab Video (Trạng thái *Mới*).

In [ ]:
GIOI_HAN = 1   # ✏️
print('Đã thêm:', zs.run(ACCOUNTS, limit=GIOI_HAN))

## Bước 2 – AI đề xuất cắt ✏️
Xử lý các video **Trạng thái = Mới** ở tab Video: tải transcript, gửi AI (nhà cung cấp/model theo tab **Cấu hình**), ghi kết quả vào tab **Chi tiết cắt**.
Sau đó Trạng thái → **Chờ duyệt** (hoặc **Đã duyệt** nếu Chế độ = Tự động). Video không có transcript sẽ bỏ qua.

In [ ]:
from sheet_job import Job, records
from ai_cut import propose_cuts
CHI_MA = ''   # ✏️ để trống = mọi video 'Mới'; hoặc ghi 1 mã, ví dụ 'IT-Team-B01'
vids = records(zs.ss.worksheet('Video').get_all_values())
todo = [v['Mã video'] for v in vids if v.get('Trạng thái') == 'Mới' and (not CHI_MA or v['Mã video'] == CHI_MA)]
print('Cần xử lý:', todo)
for ma in todo:
    job = Job(ma)
    mp4, vtt = job.find_inputs()
    if not vtt:
        print(f'⚠️ {ma}: không có transcript – bỏ qua'); continue
    path = job.download(vtt['id'], f'/content/{ma}.vtt')
    try:
        propose_cuts(job, ENV, path)
    except Exception as e:
        job.set(**{'Trạng thái': 'Lỗi', 'Ghi chú': f'AI lỗi: {str(e)[:250]}'})
        print(f'❌ {ma}: {e}')